# Notebook Overview — Evaluate Development Results

## Purpose

This notebook evaluates development-subset VideoQA results generated by previous project notebooks and produces quantitative performance metrics, analysis summaries, visualizations, and reporting artifacts.

Because NExT-QA is a multiple-choice VideoQA benchmark, multiple-choice accuracy is the primary evaluation metric when experiment outputs use `answer_mode = "multiple_choice"`. Exact-match and partial-match text comparisons are retained as secondary diagnostic metrics that provide additional insight into model-generated responses but are not considered the primary benchmark score.

During the current development phase, this notebook evaluates baseline VideoQA results generated by Notebook 01. The notebook is designed to support future comparative evaluation of baseline, pretrained-representation, and autoencoder-based VideoQA experiments using a common evaluation framework.

Evaluation procedures include prediction validation, multiple-choice accuracy analysis, reasoning-category analysis, question-type analysis, runtime analysis, visualization generation, and experiment reporting.

## Inputs

* VideoQA prediction results
* Experiment summary reports
* Runtime statistics
* NExT-QA validation annotations
* Project configuration settings

## Outputs

* Multiple-choice evaluation metrics
* Prediction verification summaries
* Choice prediction accuracy summaries
* Reasoning-category performance analyses
* Question-type performance analyses
* Answer-length analyses
* Runtime analyses
* Performance visualizations
* Evaluation reports
* Saved reporting artifacts

## Workflow

The workflow begins by loading experiment outputs and reference annotation data. Input files and required prediction columns are validated before evaluation datasets are prepared.

Prediction results are matched with NExT-QA annotation records to support reasoning-category and question-type analysis. For multiple-choice experiments, predicted answer choices are compared against ground-truth answer choices to compute benchmark accuracy metrics. Exact-match and partial-match text metrics are also computed as secondary diagnostic measures.

Evaluation metrics, answer-length statistics, runtime summaries, category-level analyses, and visualization artifacts are generated and saved for later reporting and experiment comparison.

The notebook provides a common evaluation framework that can be applied consistently across baseline, pretrained-representation, and autoencoder-based VideoQA experiments.

## Notes

This notebook does not perform VideoQA inference and does not require access to raw video files. Evaluation is performed using saved experiment outputs and NExT-QA reference annotations.

For NExT-QA multiple-choice experiments, choice accuracy is the primary benchmark metric. Exact-match and partial-match text comparisons are retained for diagnostic analysis only. Future work may incorporate semantic similarity metrics and additional benchmark measures to complement the current evaluation framework.


### 🔷 Step 1 — Initialize Evaluation Environment

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files, configuration modules, and dataset resources are available for evaluation.
* Prepare the notebook environment for loading saved experiment results and evaluation reference data.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Initialize Evaluation Environment
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = False

import os
from pathlib import Path
import pandas as pd

from google.colab import userdata, drive

print("Initializing Notebook 08 environment...")
print("-" * 60)

# ------------------------------------------------------------
# Clone Required Repository Files
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    print("\nMounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    print("\nGoogle Drive already mounted.")

# ------------------------------------------------------------
# Load Project Configuration
# ------------------------------------------------------------

print("\nLoading project configuration...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# Resolve Experiment Root (multi-experiment aware)
# ------------------------------------------------------------

EXPERIMENTS_DRIVE_DIR = GOOGLE_DRIVE_ROOT / "experiments"

required_paths = [
    Path("src"),
    QUESTIONS_DIR,
    METADATA_DIR,
    GOOGLE_DRIVE_ROOT,
    EXPERIMENTS_DRIVE_DIR,
]

missing_paths = [
    path for path in required_paths if not Path(path).exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")
    raise FileNotFoundError("One or more required project paths are missing.")

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Project paths initialized.")
print(f"Experiment root: {EXPERIMENTS_DRIVE_DIR}")

# ------------------------------------------------------------
# Discover Available Experiment Directories
# ------------------------------------------------------------

experiment_dirs = sorted(
    path for path in EXPERIMENTS_DRIVE_DIR.iterdir()
    if path.is_dir()
)

if not experiment_dirs:
    raise FileNotFoundError(
        f"No experiment directories found in: {EXPERIMENTS_DRIVE_DIR}"
    )

experiment_registry_df = pd.DataFrame({
    "experiment_name": [path.name for path in experiment_dirs],
    "experiment_dir": [str(path) for path in experiment_dirs],
})

print("\nAvailable experiment directories:")
print("-" * 60)
display(experiment_registry_df)

# ------------------------------------------------------------
# Load NExT-QA Annotation Metadata
# ------------------------------------------------------------

print("\nLoading NExT-QA annotation metadata...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata ready.")
print(f"Annotation records : {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nNotebook 08 initialization complete.")
print("-" * 60)
print("Ready to compare development experiment artifacts.")



### 🔷 Step 2 — Load ALL Experiment Evaluation Data

* Restore the configured VideoQA prediction artifacts from Google Drive to the local workspace.
* Load prediction records, prediction validation records, and experiment summary metadata produced by the upstream VideoQA notebook.
* Load NExT-QA annotations for the configured evaluation split.
* Verify that all required evaluation input artifacts are present, readable, and conform to the expected schema.
* Confirm that prediction records, validation records, summary metadata, and annotation records contain the required columns.
* Display dataset sizes, artifact locations, column information, and sample records to verify successful restoration and loading.

In [ ]:
# ============================================================
# Step 2: Load ALL Experiment Evaluation Data
# ============================================================

import pandas as pd

print("Loading ALL experiment evaluation data...\n")

all_experiment_predictions = []

for _, row in experiment_registry_df.iterrows():

    experiment_name = row["experiment_name"]
    experiment_dir = Path(row["experiment_dir"])

    print(f"Loading experiment: {experiment_name}")

    try:

        videoqa_dir = experiment_dir / "videoqa"

        if not videoqa_dir.exists():
            print(f"  ⚠ No videoqa directory: {experiment_name}")
            continue

        # ------------------------------------------------------------
        # Select correct file based on experiment type
        # ------------------------------------------------------------

        if experiment_name.startswith("qwen2vl"):
            pred_file = videoqa_dir / "baseline_predictions.csv"

        elif experiment_name.startswith("clip"):
            pred_file = videoqa_dir / "representation_videoqa_predictions.csv"

        else:
            print(f"  ⚠ Unknown experiment type: {experiment_name}")
            continue

        if not pred_file.exists():
            print(f"  ⚠ Missing predictions file: {pred_file}")
            continue

        df = pd.read_csv(pred_file)

        # ------------------------------------------------------------
        # Normalize schema
        # ------------------------------------------------------------

        df["experiment_name"] = experiment_name

        try:
            df["experiment_type"] = infer_experiment_type(experiment_name)
        except Exception:
            df["experiment_type"] = "unknown"

        all_experiment_predictions.append(df)

        print(f"  ✔ Loaded {len(df):,} records")

    except Exception as e:
        print(f"  ❌ Failed {experiment_name}: {str(e)}")

# ------------------------------------------------------------
# Combine all experiments
# ------------------------------------------------------------

if not all_experiment_predictions:
    raise ValueError("No valid experiment prediction files found.")

all_predictions_df = pd.concat(all_experiment_predictions, ignore_index=True)

# ------------------------------------------------------------
# Validate schema
# ------------------------------------------------------------

required_cols = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "experiment_name",
]

missing_cols = [c for c in required_cols if c not in all_predictions_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("\nCombined Experiment Dataset")
print("-" * 60)
print(f"Total records      : {len(all_predictions_df):,}")
print(f"Experiments loaded : {all_predictions_df['experiment_name'].nunique():,}")

display(
    all_predictions_df.groupby("experiment_name").size().reset_index(name="count")
)



### 🔷 Step 3 — Attach Annotation Metadata

* Combine restored prediction records with NExT-QA annotation metadata for the configured evaluation split.
* Construct a unified evaluation dataset containing prediction results, ground-truth information, and supporting annotation metadata.
* Verify that the evaluation dataset contains all required columns and records.
* Validate dataset integrity by checking record counts, unique videos, and prediction coverage.
* Display summary statistics, available columns, and representative evaluation records prior to computing evaluation metrics.


In [ ]:
# ============================================================
# Step 3: Attach Annotation Metadata
# ============================================================

import pandas as pd

print("Attaching annotation metadata to all experiments...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "all_predictions_df" not in globals():
    raise NameError(
        "all_predictions_df not found. Run Step 2 first."
    )

if "annotations_df" not in globals():
    raise NameError(
        "annotations_df not found. Run Step 1 first."
    )

# ------------------------------------------------------------
# Build reference table (ground truth)
# ------------------------------------------------------------

evaluation_reference_df = (
    annotations_df[
        [
            VIDEO_ID_COLUMN,
            QUESTION_COLUMN,
            "qid" if "qid" in annotations_df.columns else VIDEO_ID_COLUMN,
            "type" if "type" in annotations_df.columns else None,
            "answer" if "answer" in annotations_df.columns else None,
        ]
    ]
    .copy()
)

# Remove None columns safely
evaluation_reference_df = evaluation_reference_df[
    [c for c in evaluation_reference_df.columns if c is not None]
]

# Drop duplicates to ensure clean join
evaluation_reference_df = evaluation_reference_df.drop_duplicates(
    subset=[VIDEO_ID_COLUMN, QUESTION_COLUMN]
)

# ------------------------------------------------------------
# Merge annotations into multi-experiment predictions
# ------------------------------------------------------------

evaluation_dataset_df = all_predictions_df.merge(
    evaluation_reference_df,
    on=[VIDEO_ID_COLUMN, QUESTION_COLUMN],
    how="left"
)

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_cols = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
    "experiment_name",
]

missing_cols = [c for c in required_cols if c not in evaluation_dataset_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if evaluation_dataset_df.empty:
    raise ValueError("Evaluation dataset is empty after merge.")

# ------------------------------------------------------------
# Summary stats (IMPORTANT: grouped)
# ------------------------------------------------------------

print("Multi-Experiment Evaluation Dataset Ready")
print("-" * 60)

print(f"Total records        : {len(evaluation_dataset_df):,}")
print(f"Experiments          : {evaluation_dataset_df['experiment_name'].nunique():,}")
print(f"Unique videos        : {evaluation_dataset_df[VIDEO_ID_COLUMN].nunique():,}")

print("\nPer-experiment accuracy:")
display(
    evaluation_dataset_df.groupby("experiment_name")["choice_correct"]
    .mean()
    .reset_index(name="accuracy")
)

print("\nPreview:")
display(evaluation_dataset_df.head())



### 🔷 Step 4 — Verify Prediction Quality

* Validate prediction outputs against expected multiple-choice evaluation rules.
* Check for missing predictions, invalid predicted choices, and missing ground-truth choices.
* Confirm that predicted choices fall within the configured answer-choice set.
* Summarize prediction correctness, invalid prediction counts, and evaluation readiness.
* Display validation results before metric computation.


In [ ]:
# ============================================================
# Step 4: Verify Prediction Quality (Per-Experiment)
# ============================================================

import pandas as pd

print("Verifying prediction quality per experiment...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df not found. Run Step 3 first.")

required_cols = [
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

quality_results = []

for exp_name, df in evaluation_dataset_df.groupby("experiment_name"):

    print(f"Checking: {exp_name}")

    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{exp_name} missing columns: {missing_cols}"
        )

    prediction_count = len(df)

    missing_ground_truth = df["ground_truth_choice"].isna().sum()
    missing_prediction = df["predicted_choice"].isna().sum()

    valid_choice_values = set(range(len(CHOICE_COLUMNS)))

    invalid_ground_truth = (
        ~df["ground_truth_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()

    invalid_prediction = (
        ~df["predicted_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()

    correct = df["choice_correct"].sum()
    incorrect = prediction_count - correct

    accuracy = correct / prediction_count if prediction_count else 0.0

    quality_results.append({
        "experiment_name": exp_name,
        "records": prediction_count,
        "missing_gt": int(missing_ground_truth),
        "missing_pred": int(missing_prediction),
        "invalid_gt": int(invalid_ground_truth),
        "invalid_pred": int(invalid_prediction),
        "correct": int(correct),
        "incorrect": int(incorrect),
        "accuracy": float(accuracy),
    })

prediction_quality_df = pd.DataFrame(quality_results)

# ------------------------------------------------------------
# Global readiness check (soft, not blocking)
# ------------------------------------------------------------

total_issues = (
    prediction_quality_df["missing_gt"].sum()
    + prediction_quality_df["missing_pred"].sum()
    + prediction_quality_df["invalid_gt"].sum()
    + prediction_quality_df["invalid_pred"].sum()
)

evaluation_ready = total_issues == 0

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nPrediction Quality Summary (Per Experiment)")
print("-" * 60)

display(prediction_quality_df)

print("\nGlobal Summary")
print("-" * 60)
print(f"Total experiments   : {len(prediction_quality_df)}")
print(f"Total records       : {prediction_quality_df['records'].sum():,}")
print(f"Evaluation ready    : {evaluation_ready}")



### 🔷 Step 5 — Expand Evaluation Metrics

* Compute overall multiple-choice evaluation metrics from the prepared evaluation dataset.
* Calculate prediction counts, correct predictions, incorrect predictions, and choice accuracy.
* Generate question-type performance metrics when annotation category metadata is available.
* Generate answer-choice distribution metrics for ground-truth and predicted choices.
* Prepare metric tables for reporting, visualization, and downstream comparison.


In [ ]:
# ============================================================
# Step 5: Expand Evaluation Metrics
# ============================================================

import pandas as pd

print("Expanding evaluation metrics...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df not found. Run Step 3 first.")

# ------------------------------------------------------------
# Question-type metrics (VALID per experiment + global view)
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    question_type_metrics_df = (
        evaluation_dataset_df
        .groupby(["experiment_name", "type"])
        .agg(
            prediction_count=("choice_correct", "size"),
            correct_predictions=("choice_correct", "sum"),
        )
        .reset_index()
    )

    question_type_metrics_df["choice_accuracy"] = (
        question_type_metrics_df["correct_predictions"] /
        question_type_metrics_df["prediction_count"]
    )

else:

    question_type_metrics_df = pd.DataFrame()

# ------------------------------------------------------------
# Choice distribution (still useful global diagnostic)
# ------------------------------------------------------------

choice_distribution_df = (
    evaluation_dataset_df["predicted_choice"]
    .value_counts(dropna=False)
    .rename_axis("choice")
    .reset_index(name="predicted_count")
    .sort_values("choice")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Per-experiment summary (source of truth already from Step 4)
# ------------------------------------------------------------

experiment_summary_df = prediction_quality_df.copy()

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------

print("Per-Experiment Summary (from Step 4)")
print("-" * 60)
display(experiment_summary_df)

print("\nQuestion-Type Metrics (Experiment-aware)")
print("-" * 60)
display(question_type_metrics_df)

print("\nPrediction Distribution (Global Diagnostic)")
print("-" * 60)
display(choice_distribution_df)



### 🔷 Step 6 — Analyze Prediction Errors

* Identify incorrect prediction records for qualitative review.
* Compare ground-truth answer choices against predicted answer choices.
* Summarize errors by question type and answer-choice pattern.
* Generate an error-analysis dataset for reporting and downstream inspection.
* Display representative incorrect predictions to support interpretation of model behavior.



In [ ]:
# ============================================================
# Step 6: Analyze Prediction Errors
# ============================================================

import pandas as pd

print("Analyzing prediction errors (per experiment)...\n")

if "evaluation_dataset_df" not in globals():
    raise NameError("evaluation_dataset_df was not found. Run Step 3 first.")

# ------------------------------------------------------------
# Split correct vs incorrect (global view)
# ------------------------------------------------------------

correct_predictions_df = evaluation_dataset_df[
    evaluation_dataset_df["choice_correct"] == True
].copy()

incorrect_predictions_df = evaluation_dataset_df[
    evaluation_dataset_df["choice_correct"] == False
].copy()

# ------------------------------------------------------------
# ERROR ANALYSIS BY EXPERIMENT (KEY FIX)
# ------------------------------------------------------------

error_summary_by_experiment_df = (
    evaluation_dataset_df
    .groupby("experiment_name")
    .agg(
        total=("choice_correct", "size"),
        correct=("choice_correct", "sum"),
    )
    .reset_index()
)

error_summary_by_experiment_df["incorrect"] = (
    error_summary_by_experiment_df["total"] -
    error_summary_by_experiment_df["correct"]
)

error_summary_by_experiment_df["error_rate"] = (
    error_summary_by_experiment_df["incorrect"] /
    error_summary_by_experiment_df["total"]
)

# ------------------------------------------------------------
# ERROR TYPE BY EXPERIMENT (VERY IMPORTANT)
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    error_type_by_experiment_df = (
        incorrect_predictions_df
        .groupby(["experiment_name", "type"])
        .size()
        .reset_index(name="error_count")
        .sort_values(["experiment_name", "error_count"], ascending=[True, False])
    )

else:
    error_type_by_experiment_df = pd.DataFrame()

# ------------------------------------------------------------
# ERROR PATTERNS (global but still useful)
# ------------------------------------------------------------

error_choice_pattern_df = (
    incorrect_predictions_df
    .groupby(["ground_truth_choice", "predicted_choice"])
    .size()
    .reset_index(name="error_count")
    .sort_values("error_count", ascending=False)
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("Error Summary by Experiment")
print("-" * 60)
display(error_summary_by_experiment_df)

print("\nError Types by Experiment")
print("-" * 60)
display(error_type_by_experiment_df)

print("\nGlobal Error Patterns (Confusions)")
print("-" * 60)
display(error_choice_pattern_df)

print("\nExample Errors")
print("-" * 60)

if incorrect_predictions_df.empty:
    print("No errors found.")
else:
    display(incorrect_predictions_df.head(10))



### 🔷 Step 7 — Generate Evaluation Visualizations

* Create visual summaries of overall multiple-choice evaluation results.
* Plot question-type accuracy to compare performance across NExT-QA question categories.
* Plot ground-truth and predicted answer-choice distributions.
* Save generated figures to the evaluation output directory for reporting and documentation.
* Display generated visualizations for review inside the notebook.


In [ ]:
# ============================================================
# Step 7: Generate Evaluation Visualizations
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

print("Generating multi-experiment evaluation visualizations...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "prediction_quality_df" not in globals():
    raise NameError(
        "prediction_quality_df was not found. Run Step 4 first."
    )

if "question_type_metrics_df" not in globals():
    raise NameError(
        "question_type_metrics_df was not found. Run Step 5 first."
    )

# ------------------------------------------------------------
# Create output directories (Notebook 08 global outputs)
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

evaluation_output_root = OUTPUTS_DIR / "evaluation"
evaluation_output_root.mkdir(parents=True, exist_ok=True)

evaluation_figures_dir = evaluation_output_root / "figures"
evaluation_figures_dir.mkdir(parents=True, exist_ok=True)

generated_figures = []

# ============================================================
# 1. EXPERIMENT ACCURACY COMPARISON (PRIMARY METRIC)
# ============================================================

plt.figure(figsize=(6, 4))

plt.bar(
    prediction_quality_df["experiment_name"],
    prediction_quality_df["accuracy"],
)

plt.title("Experiment Accuracy Comparison")
plt.xlabel("Experiment")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)

plt.tight_layout()

fig1 = evaluation_figures_dir / "experiment_accuracy.png"
plt.savefig(fig1, dpi=150)
plt.show()

generated_figures.append({
    "figure": "experiment_accuracy",
    "path": str(fig1),
})

# ============================================================
# 2. QUESTION TYPE ACCURACY BY EXPERIMENT
# ============================================================

if not question_type_metrics_df.empty:

    plt.figure(figsize=(10, 5))

    for exp in question_type_metrics_df["experiment_name"].unique():

        subset = question_type_metrics_df[
            question_type_metrics_df["experiment_name"] == exp
        ]

        plt.plot(
            subset["type"].astype(str),
            subset["choice_accuracy"],
            marker="o",
            label=exp,
        )

    plt.title("Question Type Accuracy by Experiment")
    plt.xlabel("Question Type")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()

    fig2 = evaluation_figures_dir / "question_type_by_experiment.png"
    plt.savefig(fig2, dpi=150)
    plt.show()

    generated_figures.append({
        "figure": "question_type_by_experiment",
        "path": str(fig2),
    })

# ============================================================
# SAVE FIGURE INVENTORY
# ============================================================

generated_figures_df = pd.DataFrame(generated_figures)

fig_inventory = evaluation_figures_dir / "generated_figures.csv"
generated_figures_df.to_csv(fig_inventory, index=False)

print("Evaluation visualizations generated.")
print("-" * 60)
print(f"Figures created: {len(generated_figures_df)}")

display(generated_figures_df)



### 🔷 Step 8 — Save Evaluation Results + Final Selection Artifact

* Save the prepared evaluation dataset and computed metric tables to the local evaluation output directory.
* Save prediction quality checks, question-type metrics, answer-choice distributions, and error-analysis outputs.
* Verify that all expected evaluation artifacts are successfully written and reloadable.
* Preserve evaluation outputs in a standardized structure for comparison across baseline, CLIP, and autoencoder experiments.
* Display saved artifact paths for downstream review and reporting.



In [ ]:
# ============================================================
# Step 8: Save Evaluation Results + Final Selection Artifact
# ============================================================

import pandas as pd
import shutil
import json
from pathlib import Path

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_dataframes = {
    "evaluation_dataset_df": "Step 3",
    "prediction_quality_df": "Step 4",
    "question_type_metrics_df": "Step 5",
    "choice_distribution_df": "Step 5",
    "error_type_by_experiment_df": "Step 6" if "error_type_by_experiment_df" in globals() else "Step 6",
    "error_choice_pattern_df": "Step 6",
    "generated_figures_df": "Step 7",
}

missing_dataframes = [
    name for name in required_dataframes
    if name not in globals()
]

if missing_dataframes:
    raise NameError(f"Missing dataframes: {missing_dataframes}")

# ============================================================
# FIX: Notebook 08 evaluation output directory (STRICT MODE SAFE)
# ============================================================

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_OUTPUT_DIR = OUTPUTS_DIR / "evaluation"
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_OUTPUT_DRIVE_DIR = (
    GOOGLE_DRIVE_ROOT / "evaluation"
)
EVALUATION_OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Define outputs
# ------------------------------------------------------------

outputs = [
    (evaluation_dataset_df, EVALUATION_OUTPUT_DIR / "evaluation_dataset.csv"),
    (prediction_quality_df, EVALUATION_OUTPUT_DIR / "prediction_quality.csv"),
    (question_type_metrics_df, EVALUATION_OUTPUT_DIR / "question_type_metrics.csv"),
    (choice_distribution_df, EVALUATION_OUTPUT_DIR / "choice_distribution.csv"),
    (error_choice_pattern_df, EVALUATION_OUTPUT_DIR / "error_choice_pattern.csv"),
    (generated_figures_df, EVALUATION_OUTPUT_DIR / "generated_figures.csv"),
]

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

for df, path in outputs:

    df.to_csv(path, index=False)

    if not path.exists():
        raise FileNotFoundError(f"Failed to save: {path}")

# ------------------------------------------------------------
# CREATE FINAL MODEL SELECTION ARTIFACT (IMPORTANT)
# ------------------------------------------------------------

best_experiment_row = (
    prediction_quality_df
    .sort_values("accuracy", ascending=False)
    .iloc[0]
)

best_model_artifact = {
    "best_experiment": best_experiment_row["experiment_name"],
    "best_accuracy": float(best_experiment_row["accuracy"]),
    "all_experiments": prediction_quality_df.to_dict(orient="records"),
}

best_model_path = EVALUATION_OUTPUT_DIR / "best_model.json"

with open(best_model_path, "w") as f:
    json.dump(best_model_artifact, f, indent=4)

# ------------------------------------------------------------
# PROMOTE TO GOOGLE DRIVE
# ------------------------------------------------------------

EVALUATION_OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for _, row in prediction_quality_df.iterrows():
    pass  # (optional extension point)

shutil.copy2(best_model_path, EVALUATION_OUTPUT_DRIVE_DIR / best_model_path.name)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("Evaluation results saved.")
print("-" * 60)
print(f"Best experiment : {best_model_artifact['best_experiment']}")
print(f"Best accuracy   : {best_model_artifact['best_accuracy']:.3f}")
print(f"Output dir      : {EVALUATION_OUTPUT_DIR}")



### 🔷 Step 9 — Notebook Summary

* Summarize the completed development evaluation run.
* Report the configured evaluation source, dataset mode, split, prediction method, and representation sources.
* Display overall accuracy, prediction counts, question-type metrics, and error-analysis summary.
* List local and Google Drive output directories containing generated evaluation artifacts.
* Confirm that Notebook 08 outputs are ready for downstream comparison in Notebook 09.


In [ ]:
# ============================================================
# Step 9: Notebook Summary
# ============================================================

print("Notebook 08 complete.")
print("=" * 60)

# ------------------------------------------------------------
# Verify REQUIRED objects only (no optional dependencies)
# ------------------------------------------------------------

required_objects = {
    "evaluation_dataset_df": "Step 3",
    "prediction_quality_df": "Step 4",
    "question_type_metrics_df": "Step 5",
}

missing_required = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_required:
    raise NameError(
        f"Missing REQUIRED objects: {missing_required}"
    )

# ------------------------------------------------------------
# Safe optional objects (do not break notebook)
# ------------------------------------------------------------

optional_objects = [
    "incorrect_predictions_df",
    "error_type_by_experiment_df",
]

for obj in optional_objects:
    if obj not in globals():
        globals()[obj] = None

# ------------------------------------------------------------
# CORE SUMMARY METRICS (MULTI-EXPERIMENT SAFE)
# ------------------------------------------------------------

total_predictions = len(evaluation_dataset_df)
correct_predictions = int(evaluation_dataset_df["choice_correct"].sum())
incorrect_predictions = total_predictions - correct_predictions

print("\nMulti-Experiment Evaluation Summary")
print("-" * 60)

print(f"Total predictions        : {total_predictions:,}")
print(f"Experiments evaluated    : {evaluation_dataset_df['experiment_name'].nunique():,}")

# ------------------------------------------------------------
# PER-EXPERIMENT RESULTS (PRIMARY OUTPUT)
# ------------------------------------------------------------

print("\nPer-Experiment Results")
print("-" * 60)

display(
    prediction_quality_df[[
        "experiment_name",
        "records",
        "correct",
        "incorrect",
        "accuracy"
    ]]
)

# ------------------------------------------------------------
# QUESTION TYPE PERFORMANCE
# ------------------------------------------------------------

print("\nQuestion-Type Performance")
print("-" * 60)

if question_type_metrics_df is not None and not question_type_metrics_df.empty:
    display(question_type_metrics_df)
else:
    print("No question-type metrics available.")

# ------------------------------------------------------------
# ERROR SUMMARY (ONLY IF AVAILABLE)
# ------------------------------------------------------------

print("\nError Analysis Summary")
print("-" * 60)

if incorrect_predictions_df is not None:
    print(f"Incorrect predictions: {len(incorrect_predictions_df):,}")
else:
    print("Error analysis not available (optional Step 6 outputs missing).")

if error_type_by_experiment_df is not None:
    print("\nErrors by Experiment + Type")
    display(error_type_by_experiment_df)

# ------------------------------------------------------------
# FINAL MODEL SELECTION OUTPUT
# ------------------------------------------------------------

print("\nBest Model Selection (from Step 8)")
print("-" * 60)

if "best_model_path" in globals():
    print(f"Best model artifact saved at: {best_model_path}")
else:
    print("Best model artifact not available in this run.")

summary_path = EVALUATION_OUTPUT_DIR / "notebook08_summary.txt"

with open(summary_path, "w") as f:
    f.write("Notebook 08 Complete\n")
    f.write("=" * 40 + "\n")
    f.write(f"Total predictions: {total_predictions}\n")
    f.write(f"Experiments: {evaluation_dataset_df['experiment_name'].nunique()}\n")
    f.write("\nPer-experiment accuracy:\n")
    f.write(prediction_quality_df.to_string(index=False))

shutil.copy2(
    summary_path,
    EVALUATION_OUTPUT_DRIVE_DIR / summary_path.name
)

# ------------------------------------------------------------
# FINAL SYSTEM MESSAGE
# ------------------------------------------------------------

print("\nNotebook 08 Summary")
print("-" * 60)

print("✔ Multi-experiment evaluation completed")
print("✔ CLIP vs Qwen2 comparison completed")
print("✔ Metrics computed successfully")
print("✔ Notebook 08 pipeline finished")

print("\nReady for Notebook 09 (Final Full Dataset Evaluation)")

